# 05a · Descarga del conectoma GFS desde neuPrint (correr en Colab)

Etapa 5, parte que necesita el token. Descarga **el mismo subgrafo del Hito 2** (`notebooks/hito2/03_conectoma_neuprint.ipynb`): Giant Fiber System de **MaleCNS v1.0** con su capa de entrada, **327 neuronas** de los tipos `LC4`, `LPLC2`, `DNp01`, `PSI`, `TTMn`, `DLMn a, b` y `DLMn c-f`.

Diferencias con el Hito 2:
- Se guardan **todas** las propiedades de cada neurona, en particular el **neurotransmisor predicho** y su confianza, que fijan el signo de cada conexión (Opción A, revisión §12).
- El peso de cada conexión se toma del **total** de sinapsis del par (`ConnectsTo.weight`) y se compara con la suma por ROI que usaba el Hito 2.

No se calcula nada más aquí: el signo, la máscara y los hallazgos se hacen en local (`05b`), sin token.

**Cómo correrlo:** en Colab, *Archivo → Subir notebook*, y en la llave 🔑 (Secretos) debe estar `NEUPRINT_TOKEN` con el acceso al notebook activado. *Entorno de ejecución → Ejecutar todas*. Al final se descargan tres archivos: `gfs_nodes.csv`, `gfs_edges.csv` y `gfs_meta.json`. Pégalos en `data/processed/hito3/` del repositorio.

In [1]:
import os

# El token sale del secreto de Colab NEUPRINT_TOKEN (o de la variable de entorno del mismo nombre).
# Nunca se imprime ni se escribe en el notebook.
try:
    from google.colab import userdata
    os.environ["NEUPRINT_TOKEN"] = userdata.get("NEUPRINT_TOKEN")
except ImportError:
    if not os.environ.get("NEUPRINT_TOKEN"):
        raise RuntimeError("Define la variable de entorno NEUPRINT_TOKEN antes de correr el notebook.")
print("Token cargado:", bool(os.environ.get("NEUPRINT_TOKEN")))

Token cargado: True


In [2]:
!pip install -q neuprint-python

In [3]:
import json
from pathlib import Path

import neuprint
import pandas as pd
from neuprint import Client, NeuronCriteria, fetch_adjacencies, fetch_neurons

SERVER = "https://neuprint.janelia.org"
DATASET = "male-cns:v1.0"
TIPOS = ["LC4", "LPLC2", "DNp01", "PSI", "TTMn", "DLMn a, b", "DLMn c-f"]  # igual que el Hito 2
CAPA = {"LC4": "entrada", "LPLC2": "entrada", "DNp01": "decision", "PSI": "decision",
        "TTMn": "salida", "DLMn a, b": "salida", "DLMn c-f": "salida"}

client = Client(SERVER, dataset=DATASET, token=os.environ["NEUPRINT_TOKEN"])
print("Conectado a", SERVER, "| dataset", DATASET, "| neuprint-python", neuprint.__version__)

Conectado a https://neuprint.janelia.org | dataset male-cns:v1.0 | neuprint-python 0.6.4


## 1. Neuronas

Se guardan todas las propiedades escalares (se descartan solo las listas de ROI). `side` sale del sufijo `_L`/`_R` de `instance`, como en el Hito 2.

In [4]:
neuronas, _ = fetch_neurons(NeuronCriteria(type=TIPOS, regex=False))
print("Neuronas:", len(neuronas))
assert len(neuronas) == 327, "El Hito 2 tenía 327 neuronas; revisar antes de seguir."

escalares = [c for c in neuronas.columns
             if not neuronas[c].map(lambda v: isinstance(v, (list, dict, tuple))).any()]
nodes = neuronas[escalares].copy()
nodes["side"] = nodes["instance"].astype(str).str.extract(r"_(L|R)$", expand=False)
nodes["capa"] = nodes["type"].map(CAPA)
nodes = nodes.sort_values(["capa", "type", "side", "bodyId"]).reset_index(drop=True)

cols_nt = [c for c in nodes.columns if "Nt" in c or c.startswith("nt")]
print("Columnas de neurotransmisor:", cols_nt)
print("Columnas descartadas (listas):", sorted(set(neuronas.columns) - set(escalares)))
display(nodes.groupby("type").size().rename("neuronas").to_frame())

Neuronas: 327
Columnas de neurotransmisor: ['consensusNt', 'celltypeTotalNtPredictions', 'predictedNt', 'totalNtPredictions', 'predictedNtConfidence', 'celltypePredictedNtConfidence', 'celltypePredictedNt']
Columnas descartadas (listas): ['inputRois', 'outputRois', 'roiInfo', 'somaLocation']


,neuronas
type,
"DLMn a, b",2
DLMn c-f,8
DNp01,2
LC4,126
LPLC2,185
PSI,2
TTMn,2


In [5]:
# Neurotransmisor predicho por tipo (lo que fija el signo de las conexiones de salida de cada neurona).
for c in [c for c in ("predictedNt", "consensusNt", "celltypePredictedNt") if c in nodes.columns]:
    print("==", c)
    display(pd.crosstab(nodes["type"], nodes[c].fillna("(vacío)")))

== predictedNt


predictedNt,acetylcholine,histamine,unclear
type,,,
"DLMn a, b",1,0,1
DLMn c-f,1,0,7
DNp01,2,0,0
LC4,126,0,0
LPLC2,185,0,0
PSI,0,0,2
TTMn,0,1,1


== consensusNt


consensusNt,acetylcholine,glutamate,unclear
type,,,
"DLMn a, b",0,0,2
DLMn c-f,0,0,8
DNp01,2,0,0
LC4,126,0,0
LPLC2,185,0,0
PSI,0,0,2
TTMn,0,2,0


== celltypePredictedNt


celltypePredictedNt,acetylcholine,histamine,unclear
type,,,
"DLMn a, b",0,0,2
DLMn c-f,0,0,8
DNp01,2,0,0
LC4,126,0,0
LPLC2,185,0,0
PSI,0,0,2
TTMn,0,2,0


## 2. Conexiones internas

Peso total de cada par pre→post (`ConnectsTo.weight`, todas las sinapsis químicas del par). Como control se compara con la suma por ROI de `fetch_adjacencies`, que era lo que usaba el Hito 2.

In [6]:
ids = nodes["bodyId"].astype(int).tolist()
q = f"""
MATCH (a:Neuron)-[w:ConnectsTo]->(b:Neuron)
WHERE a.bodyId IN {ids} AND b.bodyId IN {ids}
RETURN a.bodyId AS bodyId_pre, b.bodyId AS bodyId_post, w.weight AS weight
"""
edges = client.fetch_custom(q)

_, por_roi = fetch_adjacencies(ids, ids)
suma_roi = por_roi.groupby(["bodyId_pre", "bodyId_post"], as_index=False)["weight"].sum()
comp = edges.merge(suma_roi, on=["bodyId_pre", "bodyId_post"], how="outer", suffixes=("_total", "_roi")).fillna(0)
print("Conexiones (pares):", len(edges), "| Hito 2 (suma por ROI):", len(suma_roi))
print("Sinapsis totales:", int(edges["weight"].sum()), "| suma por ROI:", int(suma_roi["weight"].sum()))
print("Pares con peso distinto:", int((comp["weight_total"] != comp["weight_roi"]).sum()))

tipo = nodes.set_index("bodyId")["type"]
lado = nodes.set_index("bodyId")["side"]
edges["type_pre"], edges["type_post"] = edges["bodyId_pre"].map(tipo), edges["bodyId_post"].map(tipo)
edges["side_pre"], edges["side_post"] = edges["bodyId_pre"].map(lado), edges["bodyId_post"].map(lado)
edges = edges.sort_values(["weight", "bodyId_pre", "bodyId_post"], ascending=[False, True, True]).reset_index(drop=True)
display(edges.groupby(["type_pre", "type_post"])["weight"].agg(["count", "sum"])
        .sort_values("sum", ascending=False))

  0%|          | 0/2 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Conexiones (pares): 20684 | Hito 2 (suma por ROI): 20684
Sinapsis totales: 79950 | suma por ROI: 79950
Pares con peso distinto: 0


count    sum
type_pre  type_post              
LPLC2     LPLC2      13434  46178
LC4       LC4         5334  18392
          DNp01        126   6362
LPLC2     DNp01        185   4862
          LC4         1180   2902
PSI       DLMn c-f       8    406
LC4       LPLC2        306    361
PSI       DNp01          4     95
DNp01     TTMn           2     90
DLMn c-f  DLMn c-f      26     90
DNp01     LC4           32     45
PSI       DLMn a, b      2     43
          PSI            2     22
DLMn c-f  DLMn a, b      8     21
DLMn a, b DLMn c-f       8     18
DNp01     PSI            4     16
DLMn a, b PSI            2     15
DLMn c-f  PSI            7     13
DNp01     LPLC2          8      8
DLMn a, b DLMn a, b      2      6
TTMn      DNp01          2      3
DNp01     DNp01          2      2

## 3. Exportación

In [7]:
out = Path("/content/gfs") if Path("/content").is_dir() else Path("gfs")
out.mkdir(parents=True, exist_ok=True)
nodes.to_csv(out / "gfs_nodes.csv", index=False, encoding="utf-8")
edges.to_csv(out / "gfs_edges.csv", index=False, encoding="utf-8")
meta = {
    "server": SERVER, "dataset": DATASET, "neuprint_python": neuprint.__version__,
    "fecha_descarga": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "tipos": TIPOS, "capa": CAPA, "columnas_nt": cols_nt,
    "n_neuronas": int(len(nodes)), "n_conexiones": int(len(edges)), "n_sinapsis": int(edges["weight"].sum()),
    "peso": "ConnectsTo.weight total del par (sinapsis químicas); neuPrint no trae sinapsis eléctricas",
}
(out / "gfs_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
for f in ("gfs_nodes.csv", "gfs_edges.csv", "gfs_meta.json"):
    print(f, (out / f).stat().st_size // 1024, "KB")

try:
    from google.colab import files
    for f in ("gfs_nodes.csv", "gfs_edges.csv", "gfs_meta.json"):
        files.download(str(out / f))
except ImportError:
    pass

gfs_nodes.csv 83 KB
gfs_edges.csv 588 KB
gfs_meta.json 0 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>